<a href="https://colab.research.google.com/github/MNTech955/100-days-of-machine-learning/blob/main/Machine_Learning__Without_Pipelines__A_Z.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('sample_data/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [71]:
#inplace=True->make change directly donot make the copy
df.drop(columns=['PassengerId','Name','Ticket','Cabin'], inplace=True)

# Step 1 -> train/test/splital dedia
#x means input mein only survival chor kar baqi sare column dediye , and output mein survival dedia means y
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Survived']),
    df['Survived'],
    test_size=0.2,
    random_state=42
)

X_train.head()

y_train.sample(5)

,Survived
118,0
90,0
786,1
461,0
79,1


In [72]:
#there is some missing value in the column and until we not fill it we donot go ahead
df.isnull().sum()


,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [73]:
# Applying imputation
# SimpleImputer() is a tool from scikit-learn.
# By default, it uses mean (average) to fill missing values
# “For the Age column, fill missing values with the average age.”
# strategy='most_frequent' -->Fill missing values with the most common value in that column

si_age = SimpleImputer()

si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

X_train_embarked

array([['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['C'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['C'],
       ['S'],
       ['Q'],
       ['Q'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['Q'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['Q'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
      

In [74]:
# If test data contains a new category not seen in training, it won’t crash
# Instead, it will assign all zeros
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test_embarked)

X_train_embarked

X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

X_train_transformed = np.concatenate(
    (X_train_rem, X_train_age, X_train_sex, X_train_embarked),
    axis=1
)

X_test_transformed = np.concatenate(
    (X_test_rem, X_test_age, X_test_sex, X_test_embarked),
    axis=1
)

X_test_transformed.shape

(179, 10)

In [75]:
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed, y_train)

y_pred = clf.predict(X_test_transformed)
y_pred

accuracy_score(y_test, y_pred)

# pickle is a Python module used to:

# Save objects to a file (serialization)
# Load them back later (deserialization)
import pickle
import os

# Imagine:

# ohe_sex = your trained brain 🧠
# pickle.dump = saving your brain into a file 💾
# So later you can:
# reload it without learning again

os.makedirs('models', exist_ok=True)

pickle.dump(si_age, open('models/si_age.pkl', 'wb'))
pickle.dump(si_embarked, open('models/si_embarked.pkl', 'wb'))
pickle.dump(ohe_sex, open('models/ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl', 'wb'))
pickle.dump(clf, open('models/clf.pkl', 'wb'))

In [80]:
import pickle
import numpy as np

# Load saved objects
si_age = pickle.load(open('models/si_age.pkl', 'rb'))
si_embarked = pickle.load(open('models/si_embarked.pkl', 'rb'))
ohe_sex = pickle.load(open('models/ohe_sex.pkl', 'rb'))
ohe_embarked = pickle.load(open('models/ohe_embarked.pkl', 'rb'))
clf = pickle.load(open('models/clf.pkl', 'rb'))

# Input: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked
#Create new passenger input
# Pclass = 2
# Sex = male
# Age = 31.0
# SibSp = 0
# Parch = 0
# Fare = 10.5
# Embarked = S
#dtype=object is used because the array has both numbers and text.
#reshape(1, 7) means:
#1 row, 7 columns
#Machine learning needs input in 2D format
test_input = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'], dtype=object).reshape(1, 7)

# Transform
test_input_age = si_age.transform(test_input[:, 2].reshape(1, 1))
test_input_embarked = si_embarked.transform(test_input[:, -1].reshape(1, 1))

test_input_sex = ohe_sex.transform(test_input[:, 1].reshape(1, 1))
test_input_embarked = ohe_embarked.transform(test_input_embarked)

test_input_rem = test_input[:, [0, 3, 4, 5]]
#This combines all processed parts into one final row
#[Pclass, SibSp, Parch, Fare, Age, Sex_encoded, Embarked_encoded]
#[2, 0, 0, 10.5, 31.0, 0, 1, 0, 0, 1]
test_input_transformed = np.concatenate(
    (test_input_rem, test_input_age, test_input_sex, test_input_embarked),
    axis=1
)

# Predict
print(clf.predict(test_input_transformed))

[0]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
